# QazRuFake v2: XLM-R with per-item predictions (revision experiments)

1. *Runtime → Change runtime type → **T4 GPU***.
2. Drag **`data/qazrufake.jsonl`** into the Files panel on the left (📁) and wait for the upload to finish.
3. *Runtime → **Run all***. Allow Google Drive access when asked.

The notebook trains XLM-R base (3 seeds) in 9 settings: original vs neutralised claim text, and RU / KK / RU+KK / KK fact-check only / KK unambiguous verdicts. It saves the prediction for **every test item** to Drive, so that paired bootstrap tests and subset analyses can be done afterwards. It also computes LaBSE embeddings for detecting Russian–Kazakh duplicates. Runtime is about 1.5–2 h. If Colab disconnects, re-upload `qazrufake.jsonl` and press *Run all* again: finished runs are skipped.

When it finishes, send back the two files from Google Drive: **`qazrufake_preds_v2.jsonl`** and **`qazrufake_labse.npz`**.

In [ ]:
# 1. GPU check, packages, Drive
import torch, os, json
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU: switch Runtime -> T4 GPU, then rerun')
!pip install -q transformers datasets accelerate scikit-learn sentence-transformers
from google.colab import drive
drive.mount('/content/drive')
CORPUS = next((p for p in ['qazrufake.jsonl', 'corpus_v2.jsonl'] if os.path.exists(p)), None)
assert CORPUS, 'Upload data/qazrufake.jsonl into the Files panel (left), then run again'
R = [json.loads(l) for l in open(CORPUS, encoding='utf-8')]
OUT = '/content/drive/MyDrive/qazrufake_preds_v2.jsonl'
print(len(R), 'items; predictions ->', OUT)

In [ ]:
# 2. Training settings
SETTINGS = {
    # name: (train filter, text field)
    'ru|text':      (lambda r: r['lang'] == 'ru', 'text'),
    'ru|claim':     (lambda r: r['lang'] == 'ru', 'claim'),
    'kk|text':      (lambda r: r['lang'] == 'kk', 'text'),
    'kk|claim':     (lambda r: r['lang'] == 'kk', 'claim'),
    'ru,kk|text':   (lambda r: True, 'text'),
    'kkfc|text':    (lambda r: r['fc_kk'], 'text'),
    'kkfc|claim':   (lambda r: r['fc_kk'], 'claim'),
    'kkunamb|text': (lambda r: r['unamb'], 'text'),
    'kkunamb|claim':(lambda r: r['unamb'], 'claim'),
}
SEEDS = [1, 2, 3]
MODEL = 'xlm-roberta-base'

In [ ]:
# 3. Training function: trains one setting/seed and writes a prediction for every test item (RU and KK)
import shutil, numpy as np
from datasets import Dataset
from sklearn.metrics import f1_score, accuracy_score
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding, set_seed)

def done(setting, seed):
    if not os.path.exists(OUT): return False
    key = f'"setting": "{setting}", "seed": {seed}'
    return any(key in l for l in open(OUT))

def run(setting, seed, epochs=4, lr=2e-5, bs=16, maxlen=256):
    if done(setting, seed):
        print('skip (already done):', setting, seed); return
    filt, field = SETTINGS[setting]
    set_seed(seed)
    tok = AutoTokenizer.from_pretrained(MODEL)
    def ds(rs):
        d = Dataset.from_dict({'text': [r[field] for r in rs], 'label': [r['label'] for r in rs]})
        return d.map(lambda b: tok(b['text'], truncation=True, max_length=maxlen), batched=True)
    tr = [r for r in R if r['split'] == 'train' and filt(r)]
    dv = [r for r in R if r['split'] == 'dev' and filt(r)]
    te = [r for r in R if r['split'] == 'test']
    dtr, ddv = ds(tr), ds(dv)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=2)
    def metrics(p):
        y = p.label_ids; pr = p.predictions.argmax(-1)
        return {'f1m': f1_score(y, pr, average='macro'), 'acc': accuracy_score(y, pr)}
    args = TrainingArguments(output_dir='out', learning_rate=lr, per_device_train_batch_size=bs,
        per_device_eval_batch_size=64, num_train_epochs=epochs, weight_decay=0.01,
        warmup_steps=int(0.1 * epochs * -(-len(dtr) // bs)),
        eval_strategy='epoch', save_strategy='epoch', load_best_model_at_end=True,
        metric_for_best_model='f1m', save_total_limit=1, fp16=torch.cuda.is_available(),
        report_to='none', seed=seed)
    t = Trainer(model=model, args=args, train_dataset=dtr, eval_dataset=ddv,
                data_collator=DataCollatorWithPadding(tok), compute_metrics=metrics)
    t.train()
    p = t.predict(ds(te))
    prob = torch.softmax(torch.tensor(p.predictions), -1)[:, 1].numpy()
    with open(OUT, 'a') as f:
        for r, q in zip(te, prob):
            f.write(json.dumps({'model': MODEL, 'setting': setting, 'seed': seed, 'uid': r['uid'],
                                'test': r['lang'], 'prob': round(float(q), 5)}) + '\n')
    y = np.array([r['label'] for r in te]); pr = (prob > 0.5).astype(int)
    for lang in ['ru', 'kk']:
        m = np.array([r['lang'] == lang for r in te])
        print(setting, seed, lang, 'F1m = %.3f' % f1_score(y[m], pr[m], average='macro'))
    del t, model; torch.cuda.empty_cache(); shutil.rmtree('out', ignore_errors=True)

In [ ]:
# 4. Run everything (about 1.5-2 h on T4). Safe to rerun: finished runs are skipped.
for s in SETTINGS:
    for seed in SEEDS:
        print('=' * 20, s, seed)
        run(s, seed)
print('ALL DONE')

In [ ]:
# 5. LaBSE embeddings for Russian-Kazakh duplicate detection
from sentence_transformers import SentenceTransformer
emb_model = SentenceTransformer('sentence-transformers/LaBSE', device='cuda' if torch.cuda.is_available() else 'cpu')
E = emb_model.encode([r['text'] for r in R], batch_size=64, show_progress_bar=True, normalize_embeddings=True)
np.savez_compressed('/content/drive/MyDrive/qazrufake_labse.npz', emb=E.astype('float16'), uid=np.array([r['uid'] for r in R]))
print('saved', E.shape)

In [ ]:
# 6. Quick summary (mean macro-F1 over seeds; full analysis is done offline)
import pandas as pd
P = pd.DataFrame([json.loads(l) for l in open(OUT)])
lab = {r['uid']: r['label'] for r in R}
P['y'] = P.uid.map(lab); P['pred'] = (P.prob > 0.5).astype(int)
f = P.groupby(['setting', 'seed', 'test']).apply(lambda d: f1_score(d.y, d.pred, average='macro')).rename('f1m').reset_index()
print(f.groupby(['setting', 'test']).f1m.agg(['mean', 'std', 'count']).round(3).to_string())